# Tensores

Los **tensores** son una estructura de datos especializada que es muy similar a arrays y matrices. En PyTorch, usamos tensores para codificar las entradas y salidas de un modelo, así como los parámetros del modelo.

Los tensores son similares a los `ndarrays` de NumPy, excepto que los tensores pueden ejecutarse en GPUs u otros aceleradores de hardware. De hecho, los tensores y los arrays de NumPy a menudo pueden compartir la misma memoria subyacente, eliminando la necesidad de copiar datos (ver **Puente con NumPy**). Los tensores también están optimizados para **diferenciación automática** (veremos más sobre esto más adelante en la sección de Autograd). 

Si estás familiarizado con `ndarrays`, te sentirás como en casa con la API de Tensores. Si no, ¡síguenos!

In [ ]:
import torch
import numpy as np

## Inicializar un Tensor

Los tensores pueden inicializarse de varias maneras. Echa un vistazo a los siguientes ejemplos:

### Directamente desde datos

Los tensores pueden crearse directamente desde datos. El tipo de dato se infiere automáticamente.

In [ ]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

### Desde un array NumPy

Los tensores pueden crearse desde arrays NumPy (y viceversa - ver **Puente con NumPy**).

In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

### Desde otro tensor

El nuevo tensor retiene las propiedades (forma, tipo de dato) del tensor argumento, a menos que se sobrescriban explícitamente.

In [ ]:
x_ones = torch.ones_like(x_data) # retiene las propiedades de x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # sobrescribe el tipo de dato de x_data
print(f"Random Tensor: \n {x_rand} \n")

### Con valores aleatorios o constantes

`shape` es una tupla de dimensiones del tensor. En las funciones a continuación, determina la dimensionalidad del tensor de salida.

In [ ]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

## Atributos de un Tensor

Los atributos del tensor describen su forma, tipo de dato y el dispositivo en el que se almacenan.

In [ ]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

## Operaciones sobre Tensores

Más de 1200 operaciones de tensores, incluyendo aritmética, álgebra lineal, manipulación de matrices (transposición, indexación, slicing), muestreo y más están descritas de forma exhaustiva [aquí](https://pytorch.org/docs/stable/torch.html).

Cada una de estas operaciones puede ejecutarse en la CPU y en Aceleradores como CUDA, MPS, MTIA o XPU. Si estás usando Colab, asigna un acelerador yendo a Runtime > Change runtime type > GPU.

Por defecto, los tensores se crean en la CPU. Necesitamos mover explícitamente los tensores al acelerador usando el método `.to()` (después de verificar la disponibilidad del acelerador). Ten en cuenta que copiar tensores grandes entre dispositivos puede ser costoso en términos de tiempo y memoria!

In [ ]:
# Movemos nuestro tensor al acelerador actual si está disponible
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

Prueba algunas de las operaciones de la lista. Si estás familiarizado con la API de NumPy, encontrarás la API de Tensores muy fácil de usar.

### Indexación y slicing estándar similar a numpy:

In [ ]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

### Unir tensores

Puedes usar `torch.cat` para concatenar una secuencia de tensores a lo largo de una dimensión dada. Ver también `torch.stack`, otro operador de unión de tensores que es sutilmente diferente de `torch.cat`.

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

### Operaciones aritméticas

In [ ]:
# Esto calcula la multiplicación matricial entre dos tensores. y1, y2, y3 tendrán el mismo valor
# ``tensor.T`` devuelve la transpuesta de un tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)


# Esto calcula el producto elemento por elemento. z1, z2, z3 tendrán el mismo valor
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

### Tensores de un solo elemento

Si tienes un tensor de un solo elemento, por ejemplo agregando todos los valores de un tensor en un solo valor, puedes convertirlo a un valor numérico de Python usando `item()`:

In [ ]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

### Operaciones in-place

Las operaciones que almacenan el resultado en el operando se llaman **in-place** (en el lugar). Se denotan con un sufijo `_`. Por ejemplo: `x.copy_(y)`, `x.t_()`, cambiarán `x`.

In [ ]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

**Nota:**

Las operaciones in-place ahorran algo de memoria, pero pueden ser problemáticas al calcular derivadas debido a una pérdida inmediata del historial. Por lo tanto, se desaconseja su uso.

## Puente con NumPy

Los tensores en la CPU y los arrays de NumPy pueden compartir sus ubicaciones de memoria subyacentes, y cambiar uno cambiará el otro.

### Tensor a array NumPy

In [ ]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

Un cambio en el tensor se refleja en el array NumPy.

In [ ]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

### Array NumPy a Tensor

In [ ]:
n = np.ones(5)
t = torch.from_numpy(n)

Los cambios en el array NumPy se reflejan en el tensor.

In [ ]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")